<h1>Bibliotecas</h1>
<i> - Coleta de feed de portais de notícia - necessário para o modelo entender linguagem política<br>
<i> - Coletar diariamente para formar histórico de notícias

In [3]:
import feedparser
import pandas as pd

from datetime import datetime
from pathlib import Path

<h1>Consulta RSS</h1>

<h2>Configuração dos Portais</h2>

In [4]:
# adicionar portais com a mesma estrutura
# CNN: removida pois não trouxe resultados - entender a problematica
# BBC: buscando como "GERAL" pois não tem campo de categoria no XML - Filtrar os dados na curadoria

feeds_rss = [
    {
        "portal": "AGENCIA_BRASIL",
        "categoria": "POLITICA",
        "url_feed": "https://agenciabrasil.ebc.com.br/rss/politica/feed.xml"
    },
    {
        "portal": "BBC_BRASIL",
        "categoria": "GERAL",
        "url_feed": "https://feeds.bbci.co.uk/portuguese/rss.xml"
    },
    {
        "portal": "G1_POLITICA",
        "categoria": "POLITICA",
        "url_feed": "https://g1.globo.com/rss/g1/politica/"
    },
    {
        "portal": "UOL_NOTICIAS",
        "categoria": "GERAL",
        "url_feed": "https://rss.uol.com.br/feed/noticias.xml"
    },
    {
        "portal": "PODER360",
        "categoria": "POLITICA",
        "url_feed": "https://www.poder360.com.br/feed/"
    }
]

<h2>Coleta de Dados</h2>

In [5]:
def coletar_feed_rss(feed_info):
    portal = feed_info["portal"]
    categoria = feed_info["categoria"]
    url_feed = feed_info["url_feed"]

    print(f"\nConsultando portal: {portal}")

    feed = feedparser.parse(url_feed)

    registros = []

    if feed.bozo:
        print(f"Aviso: possível problema ao ler o feed de {portal}")

    print(f"Total de notícias encontradas: {len(feed.entries)}")

    for noticia in feed.entries:
        registros.append({
            "portal": portal,
            "categoria": categoria,
            "titulo": noticia.get("title", ""),
            "link": noticia.get("link", ""),
            "resumo": noticia.get("summary", ""),
            "data_publicacao": noticia.get("published", ""),
            "url_feed": url_feed,
            "data_coleta": datetime.now().strftime("%d/%m/%Y %H:%M:%S")
        })

    return registros

registros = []

for feed_info in feeds_rss:
    registros_feed = coletar_feed_rss(feed_info)
    registros.extend(registros_feed)

df_raw = pd.DataFrame(registros)

print(f"\nTotal consolidado de notícias: {len(df_raw)}")

df_raw.head()


Consultando portal: AGENCIA_BRASIL
Total de notícias encontradas: 10

Consultando portal: BBC_BRASIL
Total de notícias encontradas: 40

Consultando portal: G1_POLITICA
Total de notícias encontradas: 100

Consultando portal: UOL_NOTICIAS
Total de notícias encontradas: 15

Consultando portal: PODER360
Total de notícias encontradas: 10

Total consolidado de notícias: 175


,portal,categoria,titulo,link,resumo,data_publicacao,url_feed,data_coleta
0,AGENCIA_BRASIL,POLITICA,"""Ninguém respeita lambe-botas"", diz Lula sobre...",https://agenciabrasil.ebc.com.br/politica/noti...,"<p><p style=""text-align: center;""><a class="""" ...","Fri, 08 May 2026 19:32:00 -0300",https://agenciabrasil.ebc.com.br/rss/politica/...,10/05/2026 01:58:55
1,AGENCIA_BRASIL,POLITICA,Dosimetria: Alcolumbre promulga lei que benefi...,https://agenciabrasil.ebc.com.br/politica/noti...,"<p><p style=""text-align: center;""><a class="""" ...","Fri, 08 May 2026 14:20:00 -0300",https://agenciabrasil.ebc.com.br/rss/politica/...,10/05/2026 01:58:55
2,AGENCIA_BRASIL,POLITICA,Especialistas e municípios criticam PL sobre m...,https://agenciabrasil.ebc.com.br/politica/noti...,"<p><p style=""text-align: center;""><a class="""" ...","Thu, 07 May 2026 16:29:00 -0300",https://agenciabrasil.ebc.com.br/rss/politica/...,10/05/2026 01:58:55
3,AGENCIA_BRASIL,POLITICA,Câmara aprova MP que prevê renovação automátic...,https://agenciabrasil.ebc.com.br/politica/noti...,"<p><p style=""text-align: center;""><a class="""" ...","Thu, 07 May 2026 14:20:00 -0300",https://agenciabrasil.ebc.com.br/rss/politica/...,10/05/2026 01:58:55
4,AGENCIA_BRASIL,POLITICA,Câmara permite usar fundos de minerais crítico...,https://agenciabrasil.ebc.com.br/politica/noti...,"<p><p style=""text-align: center;""><a class="""" ...","Thu, 07 May 2026 13:47:00 -0300",https://agenciabrasil.ebc.com.br/rss/politica/...,10/05/2026 01:58:55


<h2>Extração</h2>

In [6]:
nome_pipeline = "pipeline_noticias_reais"
nome_base = "rss_noticias_reais"
df_exportar = df_raw

data_agora = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

pasta_raw = Path(f"../dados/{nome_pipeline}/raw")
pasta_raw.mkdir(parents=True, exist_ok=True)

caminho_saida = pasta_raw / f"{nome_base}_raw_{data_agora}.csv"

df_exportar.to_csv(
    caminho_saida,
    index=False,
    encoding="utf-8-sig"
)

print(f"Arquivo bruto salvo em: {caminho_saida}")

print(f"\nTotal de registros extraídos: {len(df_exportar)}")
print(f"\nData e hora da extração: {datetime.now().strftime('%d/%m/%Y %H:%M:%S')}")

Arquivo bruto salvo em: ..\dados\pipeline_noticias_reais\raw\rss_noticias_reais_raw_2026-05-10_01-58-57.csv

Total de registros extraídos: 175

Data e hora da extração: 10/05/2026 01:58:57


In [7]:
df_raw["portal"].value_counts()

portal
G1_POLITICA       100
BBC_BRASIL         40
UOL_NOTICIAS       15
AGENCIA_BRASIL     10
PODER360           10
Name: count, dtype: int64